# 교통안전 EPDO 통합 분석 — 1부 (STEP 00 ~ STEP 07)

COMPAS 환경 실행용 (BASE_DIR = os.getcwd())
이 노트북 완료 후 **2부**를 실행하세요.

In [ ]:
import csv, json, math, os, warnings
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, roc_auc_score)
from sklearn.model_selection import train_test_split
import geopandas as gpd
from shapely.geometry import Point

try:
    import statsmodels.api as sm
    HAS_STATSMODELS = True
except ImportError:
    from sklearn.linear_model import PoissonRegressor
    HAS_STATSMODELS = False

warnings.filterwarnings("ignore", category=UserWarning)

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
# COMPAS 구조: 노트북과 같은 위치에 data/ 폴더가 있음
BASE_DIR   = os.getcwd()          # 노트북 실행 위치 (data/ 폴더의 부모)
EPDO_DIR   = os.getcwd()          # 분석 결과 저장 위치
OUTPUT_DIR = os.path.join(EPDO_DIR, "epdo_analysis", "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"BASE_DIR  : {BASE_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

---

## STEP 00 - 전처리 (원시 데이터 → 분석용 파일 생성)

원시 COMPAS 데이터로부터 STEP 01~11에 필요한 중간 파일을 생성합니다.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os, csv, warnings
from collections import defaultdict, Counter

warnings.filterwarnings("ignore")
CRS_GEO  = "EPSG:4326"
CRS_PROJ = "EPSG:5186"
DATA_DIR  = os.path.join(BASE_DIR, "data")

def find_col(df, *kws):
    cl = {c.lower().replace(" ","").replace("_",""): c for c in df.columns}
    for kw in kws:
        kw2 = kw.lower().replace(" ","").replace("_","")
        for k, v in cl.items():
            if kw2 in k: return v
    return None

def to_gdf(csv_path):
    df = pd.read_csv(csv_path, encoding="utf-8-sig", low_memory=False)
    lat = find_col(df, "위도","lat","latitude","ycoord","y좌표")
    lon = find_col(df, "경도","lon","longitude","xcoord","x좌표")
    if not lat or not lon:
        print(f"  좌표 탐지 실패: {os.path.basename(csv_path)}  컬럼: {list(df.columns[:8])}")
        return None
    df[lat] = pd.to_numeric(df[lat], errors="coerce")
    df[lon] = pd.to_numeric(df[lon], errors="coerce")
    df = df.dropna(subset=[lat, lon])
    src = CRS_PROJ if df[lat].median() > 1000 else CRS_GEO
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df[lon], df[lat]), crs=src)
    return gdf.to_crs(CRS_PROJ)

print("=" * 60)
print("STEP 00 - 원시 데이터 전처리")
print("=" * 60)

# 1. 기본 데이터 로드
print("\n[1] 격자 / 도로망 / 사고 데이터 로드...")
grids = gpd.read_file(os.path.join(DATA_DIR, "01._격자_(4개_시·구).geojson")).to_crs(CRS_PROJ)
gc = find_col(grids, "gid") or grids.columns[0]
grids = grids.rename(columns={gc: "gid"})
print(f"    격자: {len(grids)}개")

roads = gpd.read_file(os.path.join(DATA_DIR, "08.상세도로망_네트워크.geojson")).to_crs(CRS_PROJ)
print(f"    링크: {len(roads)}개  컬럼: {[c for c in roads.columns if c != 'geometry']}")

acc_raw = gpd.read_file(os.path.join(DATA_DIR, "13._교통사고이력.geojson"))
print(f"    사고: {len(acc_raw)}건  컬럼: {[c for c in acc_raw.columns if c != 'geometry']}")
acc = acc_raw.to_crs(CRS_PROJ)

uid_c  = find_col(acc, "uid","번호","사고번호") or acc.columns[0]
svr_c  = find_col(acc, "사상자","svrity","injury","피해구분","중증도","severity")
yr_c   = find_col(acc, "발생년","년도","acc_yr","year")
mon_c  = find_col(acc, "발생월","acc_mon","month")
time_c = find_col(acc, "발생시","시각","acc_time","hour","time")
week_c = find_col(acc, "요일","week")
type_c = find_col(acc, "사고유형","acc_type","type")
viol_c = find_col(acc, "법규위반","violation","viol")
print(f"    매핑: uid={uid_c}, 심각도={svr_c}, 년={yr_c}, 월={mon_c}, 요일={week_c}")

SVRITY_KEYS = ["사망","중상","경상","부상신고","상해없음","기타불명"]
WEEKEND = {"토","토요일","일","일요일","sat","sun","saturday","sunday"}

def norm_svr(v):
    v = str(v).strip()
    for k in SVRITY_KEYS:
        if k in v: return k
    return v

def norm_week(v):
    return "주말" if str(v).strip().lower() in WEEKEND else "평일"

# 2. 교통사고 → 링크 매핑
print("\n[2] 교통사고 → 링크 매핑 (sjoin_nearest)...")
acc_r = acc.reset_index(drop=True)
road_keep = [c for c in ["link_id","road_name","road_rank","geometry"] if c in roads.columns or c == "geometry"]
mapped = gpd.sjoin_nearest(acc_r, roads[road_keep].reset_index(drop=True), how="left", distance_col="_d")
mapped["_lon"] = acc_raw.reset_index(drop=True).geometry.x
mapped["_lat"] = acc_raw.reset_index(drop=True).geometry.y

def gv(row, col, default=""):
    return row[col] if col and col in row.index and pd.notna(row[col]) else default

out_link = []
for _, r in mapped.iterrows():
    out_link.append({
        "uid":           gv(r, uid_c),
        "link_id":       gv(r, "link_id"),
        "road_name":     gv(r, "road_name"),
        "injury_svrity": norm_svr(gv(r, svr_c)) if svr_c else "",
        "epdo_score":    0,
        "acc_yr":        gv(r, yr_c),
        "acc_mon":       gv(r, mon_c),
        "acc_time":      gv(r, time_c),
        "week_type":     norm_week(gv(r, week_c)) if week_c else "평일",
        "acc_type":      gv(r, type_c),
        "violation":     gv(r, viol_c),
        "road_type":     gv(r, "road_rank"),
        "lon":           r["_lon"],
        "lat":           r["_lat"],
    })

OUT_LINK = os.path.join(OUTPUT_DIR, "00_교통사고_링크매핑.csv")
with open(OUT_LINK, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(out_link[0].keys()))
    w.writeheader(); w.writerows(out_link)
print(f"    저장: {OUT_LINK} ({len(out_link):,}건)")
print(f"    심각도: {dict(Counter(r['injury_svrity'] for r in out_link).most_common(6))}")

# 3. 교통사고 → 격자 매핑
print("\n[3] 교통사고 → 격자 매핑...")
ag = gpd.sjoin(acc_r, grids[["gid","geometry"]], how="left", predicate="within")
ag_gid = ag["gid"].groupby(ag.index).first()  # deduplicate boundary points
acc_out = acc_raw.copy()
acc_out["grid_gid"] = ag_gid.reindex(acc_out.index).values
if svr_c:
    acc_out["injury_svrity"] = acc_out[svr_c].apply(norm_svr)
elif "injury_svrity" not in acc_out.columns:
    acc_out["injury_svrity"] = ""
OUT_GJ = os.path.join(OUTPUT_DIR, "00_교통사고_격자매핑.geojson")
acc_out.to_file(OUT_GJ, driver="GeoJSON")
print(f"    저장: {OUT_GJ}")

# 4. 격자별 인프라 통합
print("\n[4] 격자별 인프라 통합...")
res = pd.DataFrame({"grid_gid": grids["gid"].values})

FACILS = [
    ("18._횡단보도_위치정보.csv",  "crosswalk_cnt"),
    ("14._어린이보호구역.csv",      "child_zone_cnt"),
    ("15._학교현황.csv",            "school_cnt"),
    ("16._유치원현황.csv",          "kindergarten_cnt"),
    ("17._어린이집현황.csv",        "daycare_cnt"),
    ("19._버스정류장_위치정보.csv", "bus_stop_cnt"),
    ("20._CCTV_현황.csv",           "cctv_cnt"),
]
for fname, col in FACILS:
    fp = os.path.join(DATA_DIR, fname)
    if not os.path.exists(fp):
        res[col] = 0; print(f"  없음: {fname}"); continue
    gdf = to_gdf(fp)
    if gdf is None:
        res[col] = 0; continue
    j = gpd.sjoin(gdf, grids[["gid","geometry"]], how="left", predicate="within")
    agg = j.groupby("gid").size().reset_index().rename(columns={"gid":"grid_gid", 0: col})
    res = res.merge(agg, on="grid_gid", how="left")
    res[col] = res[col].fillna(0).astype(int)
    print(f"  {fname}: {res[col].sum():.0f}개")

# CCTV 대수
fp = os.path.join(DATA_DIR, "20._CCTV_현황.csv")
if os.path.exists(fp):
    df_cc = pd.read_csv(fp, encoding="utf-8-sig", low_memory=False)
    cam_c = find_col(df_cc, "대수","카메라","설치대수","cam")
    gdf_cc = to_gdf(fp)
    if gdf_cc is not None and cam_c:
        gdf_cc[cam_c] = pd.to_numeric(gdf_cc[cam_c], errors="coerce").fillna(1)
        j = gpd.sjoin(gdf_cc[["geometry", cam_c]], grids[["gid","geometry"]], how="left", predicate="within")
        agg = j.groupby("gid")[cam_c].sum().reset_index().rename(columns={"gid":"grid_gid", cam_c:"cctv_cam_cnt"})
        res = res.merge(agg, on="grid_gid", how="left")
    else:
        res["cctv_cam_cnt"] = res["cctv_cnt"]
else:
    res["cctv_cam_cnt"] = res["cctv_cnt"]
res["cctv_cam_cnt"] = res["cctv_cam_cnt"].fillna(0).astype(int)

# 유치원 원아수
fp = os.path.join(DATA_DIR, "16._유치원현황.csv")
if os.path.exists(fp):
    df_kg = pd.read_csv(fp, encoding="utf-8-sig", low_memory=False)
    cc = find_col(df_kg, "원아수","학생수","아동수","정원","child","student")
    gdf_kg = to_gdf(fp)
    if gdf_kg is not None and cc:
        gdf_kg[cc] = pd.to_numeric(gdf_kg[cc], errors="coerce").fillna(0)
        j = gpd.sjoin(gdf_kg[["geometry", cc]], grids[["gid","geometry"]], how="left", predicate="within")
        agg = j.groupby("gid")[cc].sum().reset_index().rename(columns={"gid":"grid_gid", cc:"kindergarten_child_cnt"})
        res = res.merge(agg, on="grid_gid", how="left")
    else:
        res["kindergarten_child_cnt"] = res["kindergarten_cnt"]
else:
    res["kindergarten_child_cnt"] = res["kindergarten_cnt"]
res["kindergarten_child_cnt"] = res["kindergarten_child_cnt"].fillna(0).astype(int)

# 과속방지턱 (포인트 파일 → sjoin으로 격자 집계)
fp = os.path.join(DATA_DIR, "21.1_과속방지턱_격자매핑.csv")
if os.path.exists(fp):
    df_sb = pd.read_csv(fp, encoding="utf-8-sig", low_memory=False)
    print(f"  과속방지턱 컬럼: {list(df_sb.columns)}")
    hgt_c = find_col(df_sb, "fac_hght","height","hght","높이")
    gdf_sb = to_gdf(fp)
    if gdf_sb is not None:
        if hgt_c:
            gdf_sb[hgt_c] = pd.to_numeric(gdf_sb[hgt_c], errors="coerce").fillna(0)
            gdf_sb["_hght_below"] = gdf_sb[hgt_c].apply(lambda x: x if x <= 10 else 0)
            gdf_sb["_hght_above"] = gdf_sb[hgt_c].apply(lambda x: x if x > 10 else 0)
        else:
            gdf_sb["_hght_below"] = gdf_sb["_hght_above"] = 0
        gdf_sb["_cnt"] = 1
        j = gpd.sjoin(gdf_sb[["geometry","_cnt","_hght_below","_hght_above"]], grids[["gid","geometry"]], how="left", predicate="within")
        agg_sb = j.groupby("gid")[["_cnt","_hght_below","_hght_above"]].sum().reset_index()
        agg_sb = agg_sb.rename(columns={"gid":"grid_gid","_cnt":"speedbump_cnt","_hght_below":"speedbump_hght_below","_hght_above":"speedbump_hght_above"})
        res = res.merge(agg_sb, on="grid_gid", how="left")
        print(f"  과속방지턱: {int(res['speedbump_cnt'].sum())}개")
    else:
        res["speedbump_cnt"] = res["speedbump_hght_below"] = res["speedbump_hght_above"] = 0
else:
    res["speedbump_cnt"] = res["speedbump_hght_below"] = res["speedbump_hght_above"] = 0
for c in ["speedbump_cnt","speedbump_hght_below","speedbump_hght_above"]:
    res[c] = res[c].fillna(0).astype(int)

OUT_INFRA = os.path.join(OUTPUT_DIR, "00_격자별_통합데이터.csv")
res.to_csv(OUT_INFRA, index=False, encoding="utf-8-sig")
print(f"\n    저장: {OUT_INFRA}  ({len(res):,}개 격자)")

# 5. 링크별 사고 집계
print("\n[5] 링크별 사고 집계...")
link_agg = defaultdict(list)
for r in out_link:
    lid = str(r.get("link_id",""))
    if lid and lid not in ("","nan"):
        link_agg[lid].append(str(r.get("uid","")))

OUT_AGG = os.path.join(OUTPUT_DIR, "00_링크별_사고집계.csv")
with open(OUT_AGG, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f)
    w.writerow(["link_id","accident_cnt","accident_uids"])
    for lid, uids in link_agg.items():
        w.writerow([lid, len(uids), "|".join(uids)])
print(f"    저장: {OUT_AGG} ({len(link_agg):,}개 링크)")

print("\n" + "="*60)
print("STEP 00 완료")
print("="*60)


---

## 01_EPDO_SCORE

# STEP 01 - 사고별 EPDO 점수 부여

- 입력: output/13._교통사고_링크매핑.csv
- 출력: epdo_analysis/output/01_사고별_EPDO점수.csv

EPDO 가중치 출처:
  이상엽(2019). 교통사고 원인행위의 벌점 추정에 관한 연구.
  대한교통학회지, 37(5), 365-378.
  사망=391, 중상=69, 경상=8, 부상신고=6, 물적피해(PDO)=1

In [ ]:
import csv
import os

INPUT_PATH  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "00_교통사고_링크매핑.csv")
OUTPUT_PATH = os.path.join(EPDO_DIR, "epdo_analysis", "output", "01_사고별_EPDO점수.csv")

EPDO_WEIGHTS = {
    "사망":   391,
    "중상":    69,
    "경상":     8,
    "부상신고":  6,
    "상해없음":  1,
    "기타불명":  8,   # 경상과 동일 (보수적 처리)
    "":          0,   # 차량단독·피해자 없음
}

In [ ]:
print("=" * 60)
print("STEP 01 - 사고별 EPDO 점수 부여")
print("=" * 60)

with open(INPUT_PATH, "r", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

print(f"\n입력 사고 수: {len(rows):,}건")

result = []
unknown = {}
for row in rows:
    svrity = row.get("injury_svrity", "").strip()
    score = EPDO_WEIGHTS.get(svrity)
    if score is None:
        unknown[svrity] = unknown.get(svrity, 0) + 1
        score = 0
    result.append({
        "uid":           row["uid"],
        "link_id":       row["link_id"],
        "road_name":     row["road_name"],
        "injury_svrity": svrity,
        "epdo_score":    score,
        "acc_yr":        row["acc_yr"],
        "acc_mon":       row["acc_mon"],
        "acc_time":      row["acc_time"],
        "week_type":     row["week_type"],
        "acc_type":      row["acc_type"],
        "violation":     row["violation"],
        "road_type":     row.get("road_rank", ""),
        "lon":           row["lon"],
        "lat":           row["lat"],
    })

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys())
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

# 검증 출력
from collections import Counter
dist = Counter(r["injury_svrity"] for r in result)
epdo_by_svrity = {}
for r in result:
    k = r["injury_svrity"]
    epdo_by_svrity[k] = epdo_by_svrity.get(k, 0) + r["epdo_score"]

total_epdo = sum(r["epdo_score"] for r in result)

print(f"\n[심각도별 건수 및 EPDO 합계]")
for k, cnt in dist.most_common():
    print(f"  {k or '(빈값)':10s}: {cnt:5d}건 | EPDO {epdo_by_svrity.get(k,0):,}점")

print(f"\n전체 EPDO 합계: {total_epdo:,}점")
print(f"저장: {OUTPUT_PATH}")

if unknown:
    print(f"\n⚠ 미정의 값(0점 처리): {unknown}")

print("=" * 60)

---

## 02_LINK_TRAFFIC

# STEP 02 - 링크별 교통량 집계

- 입력: data/08.상세도로망_네트워크.geojson (up_v_link, dw_v_link → link_id 매핑)
        data/10._추정교통량.csv (v_link_id, ALL_AADT, k_length)
- 출력: epdo_analysis/output/02_링크별_교통량.csv

노출량(exposure) = ALL_AADT_total × k_length (대·km/일)

In [ ]:
import csv
import json
import os
from collections import defaultdict

ROAD_FILE   = os.path.join(BASE_DIR, "data", "08.상세도로망_네트워크.geojson")
TRAFFIC_FILE = os.path.join(BASE_DIR, "data", "10._추정교통량.csv")
OUTPUT_PATH = os.path.join(EPDO_DIR, "epdo_analysis", "output", "02_링크별_교통량.csv")

In [ ]:
print("=" * 60)
print("STEP 02 - 링크별 교통량 집계")
print("=" * 60)

# 1. 08번에서 v_link_id → link_id 역매핑
print("\n[1] 도로망 로드 중...")
with open(ROAD_FILE, "r", encoding="utf-8") as f:
    road_data = json.load(f)

vlink_to_link = {}   # v_link_id(str) → link_id(str)
link_props    = {}   # link_id(str) → {road_name, road_rank, length, k_length}
for feat in road_data["features"]:
    p = feat["properties"]
    lid = str(p["link_id"])
    link_props[lid] = {
        "road_name": p.get("road_name", ""),
        "road_rank": p.get("road_rank", ""),
        "length":    p.get("length", ""),
    }
    up = p.get("up_v_link")
    dw = p.get("dw_v_link")
    if up:
        vlink_to_link[str(up)] = lid
    if dw:
        vlink_to_link[str(dw)] = lid

print(f"    링크 수: {len(link_props):,}")
print(f"    v_link 매핑 수: {len(vlink_to_link):,}")

# 2. 10번 교통량 집계 (timeslot별 ALL_AADT 합산)
print("\n[2] 교통량 데이터 로드 중...")
aadt_by_vlink  = defaultdict(float)
klen_by_vlink  = {}
rank_by_vlink  = {}

with open(TRAFFIC_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        vid = row["v_link_id"]
        try:
            aadt_by_vlink[vid] += float(row["ALL_AADT"])
        except (ValueError, KeyError):
            pass
        klen_by_vlink[vid]  = row.get("k_length", "")
        rank_by_vlink[vid]  = row.get("road_rank", "")

print(f"    v_link_id 수: {len(aadt_by_vlink):,}")

# 3. v_link_id → link_id 변환 후 상행/하행 평균
link_aadt = defaultdict(list)   # link_id → [aadt_values]
link_klen = {}

for vid, aadt in aadt_by_vlink.items():
    lid = vlink_to_link.get(vid)
    if lid:
        link_aadt[lid].append(aadt)
        if lid not in link_klen:
            link_klen[lid] = klen_by_vlink.get(vid, "")

print(f"    link_id 매칭 수: {len(link_aadt):,} / {len(link_props):,}")

# 4. 링크별 교통량 산출 (상행·하행 평균)
result = []
for lid, aadt_list in link_aadt.items():
    all_aadt_total = sum(aadt_list) / len(aadt_list)   # 방향 평균
    try:
        k_length = float(link_klen.get(lid, 0))
    except ValueError:
        k_length = 0.0
    exposure = all_aadt_total * k_length   # 대·km/일

    props = link_props.get(lid, {})
    result.append({
        "link_id":       lid,
        "road_name":     props.get("road_name", ""),
        "road_rank":     props.get("road_rank", ""),
        "length_km":     props.get("length", ""),
        "k_length":      k_length,
        "ALL_AADT_total": round(all_aadt_total, 2),
        "exposure":      round(exposure, 4),
    })

result.sort(key=lambda x: -x["exposure"])

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys())
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

print(f"\n[결과] 교통량 산출 링크: {len(result):,}개")
print(f"       노출량 상위 5개:")
for r in result[:5]:
    print(f"  link_id={r['link_id']} | {r['road_name']} | AADT={r['ALL_AADT_total']:,.0f} | 노출량={r['exposure']:,.1f}")
print(f"\n저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 03_LINK_EPDO_RISK

# STEP 03 - 링크별 위험도 산출

- 입력: epdo_analysis/output/01_사고별_EPDO점수.csv
        epdo_analysis/output/02_링크별_교통량.csv
        output/13._링크별_사고집계.csv (accident_uids 참조)
- 출력: epdo_analysis/output/03_링크별_위험도.csv

위험도(epdo_rate) = epdo_total / exposure × 1,000,000
  (단위: 백만 대·km당 EPDO, 교통량 미매칭 링크는 null)

In [ ]:
import csv
import os
from collections import defaultdict

EPDO_FILE     = os.path.join(EPDO_DIR, "epdo_analysis", "output", "01_사고별_EPDO점수.csv")
TRAFFIC_FILE  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "02_링크별_교통량.csv")
AGG_FILE      = os.path.join(EPDO_DIR, "epdo_analysis", "output", "00_링크별_사고집계.csv")
OUTPUT_PATH   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "03_링크별_위험도.csv")

In [ ]:
print("=" * 60)
print("STEP 03 - 링크별 위험도 산출")
print("=" * 60)

# 1. 사고별 EPDO 집계
print("\n[1] 사고별 EPDO 로드 중...")
link_epdo    = defaultdict(float)
link_cnt     = defaultdict(int)
link_meta    = {}   # link_id → {road_name, road_rank, max_speed, length}

with open(EPDO_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        lid = str(row["link_id"])
        link_epdo[lid] += float(row["epdo_score"])
        link_cnt[lid]  += 1
        if lid not in link_meta:
            link_meta[lid] = {
                "road_name": row["road_name"],
                "road_rank": row["road_type"],   # road_type에 road_rank 저장됨
            }

print(f"    사고 매핑 링크 수: {len(link_epdo):,}")

# 2. 교통량 로드
print("\n[2] 교통량 로드 중...")
traffic = {}
with open(TRAFFIC_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        lid = str(row["link_id"])
        traffic[lid] = {
            "ALL_AADT_total": float(row["ALL_AADT_total"]),
            "exposure":       float(row["exposure"]),
            "k_length":       row["k_length"],
            "length_km":      row["length_km"],
            "road_rank":      row["road_rank"],
            "road_name":      row["road_name"],
        }

# 3. accident_uids 로드
uids_by_link = {}
with open(AGG_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        uids_by_link[str(row["link_id"])] = row.get("accident_uids", "")

# 4. 링크별 위험도 산출
result = []
matched = 0
for lid, epdo_total in link_epdo.items():
    acc_cnt   = link_cnt[lid]
    epdo_avg  = round(epdo_total / acc_cnt, 4) if acc_cnt else 0
    meta      = link_meta.get(lid, {})
    traf      = traffic.get(lid)

    if traf and traf["exposure"] > 0:
        epdo_rate = round(epdo_total / traf["exposure"] * 1_000_000, 4)
        matched += 1
    else:
        epdo_rate = None

    result.append({
        "link_id":        lid,
        "road_name":      traf["road_name"] if traf else meta.get("road_name", ""),
        "road_rank":      traf["road_rank"] if traf else meta.get("road_rank", ""),
        "length_km":      traf["length_km"] if traf else "",
        "accident_cnt":   acc_cnt,
        "epdo_total":     round(epdo_total, 2),
        "epdo_avg":       epdo_avg,
        "ALL_AADT_total": round(traf["ALL_AADT_total"], 2) if traf else "",
        "exposure":       round(traf["exposure"], 4) if traf else "",
        "epdo_rate":      epdo_rate,
        "accident_uids":  uids_by_link.get(lid, ""),
    })

# epdo_rate 기준 정렬 (null은 후순위)
result.sort(key=lambda x: (x["epdo_rate"] is None, -(x["epdo_rate"] or 0)))

# 순위 부여 (epdo_rate 있는 것만)
rank = 1
for r in result:
    if r["epdo_rate"] is not None:
        r["epdo_rank"] = rank
        rank += 1
    else:
        r["epdo_rank"] = ""

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = ["link_id", "road_name", "road_rank", "length_km",
        "accident_cnt", "epdo_total", "epdo_avg",
        "ALL_AADT_total", "exposure", "epdo_rate", "epdo_rank",
        "accident_uids"]
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

print(f"\n[결과]")
print(f"    사고 발생 링크: {len(result):,}개")
print(f"    교통량 매칭 링크: {matched:,}개 ({matched/len(result)*100:.1f}%)")
print(f"\n    epdo_rate 상위 10개 (백만 대·km당 EPDO):")
for r in result[:10]:
    print(f"  rank={r['epdo_rank']:>4} | {r['road_name']:20s} | "
          f"사고{r['accident_cnt']:3d}건 | epdo_total={r['epdo_total']:>8.1f} | "
          f"rate={r['epdo_rate']:>10.2f}")
print(f"\n저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 04_CAUSE_ANALYSIS

# STEP 04 - 위험 도로 원인 분석

- 입력: epdo_analysis/output/01_사고별_EPDO점수.csv
        epdo_analysis/output/03_링크별_위험도.csv
- 출력: epdo_analysis/output/04_위험도로_원인분석.csv

선정 기준: epdo_rate 기준 상위 20개 (사고 3건 이상 링크만 대상)
  - 사고 1~2건 링크는 노출량이 작아 rate가 극단적으로 높아지므로 제외
분석 항목: violation, acc_type, road_type(road_rank), acc_time, week_type

In [ ]:
import csv
import os
from collections import Counter, defaultdict

EPDO_FILE    = os.path.join(EPDO_DIR, "epdo_analysis", "output", "01_사고별_EPDO점수.csv")
RISK_FILE    = os.path.join(EPDO_DIR, "epdo_analysis", "output", "03_링크별_위험도.csv")
OUTPUT_PATH  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "04_위험도로_원인분석.csv")

MIN_ACCIDENT = 3    # 최소 사고 건수 기준
TOP_N        = 20   # 분석 대상 상위 링크 수


def top_item(counter, n=3):
    return " | ".join(f"{k}({v}건)" for k, v in counter.most_common(n))

print("=" * 60)
print("STEP 04 - 위험 도로 원인 분석")
print("=" * 60)

# 1. 위험도 로드 → 상위 링크 선정
print(f"\n[1] 위험도 로드 중 (사고 {MIN_ACCIDENT}건 이상, 상위 {TOP_N}개)...")
risk_rows = []
with open(RISK_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        if row["epdo_rate"] and int(row["accident_cnt"]) >= MIN_ACCIDENT:
            risk_rows.append(row)

risk_rows.sort(key=lambda x: -float(x["epdo_rate"]))
top_links = risk_rows[:TOP_N]
top_link_ids = {r["link_id"] for r in top_links}
print(f"    대상 링크: {len(top_links)}개")

# 2. 사고 데이터 로드
print("\n[2] 사고 데이터 로드 중...")
accidents_by_link = defaultdict(list)
all_accidents = []

with open(EPDO_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        all_accidents.append(row)
        if row["link_id"] in top_link_ids:
            accidents_by_link[row["link_id"]].append(row)

# 3. 전체 평균 분포 (비교 기준)
all_violation = Counter(r["violation"] for r in all_accidents)
all_acc_type  = Counter(r["acc_type"]  for r in all_accidents)
all_time      = Counter(r["acc_time"]  for r in all_accidents)
all_week      = Counter(r["week_type"] for r in all_accidents)
total_all     = len(all_accidents)

# 4. 링크별 원인 분석
print("\n[3] 원인 분석 중...")
result = []

for r in top_links:
    lid  = r["link_id"]
    accs = accidents_by_link[lid]
    cnt  = len(accs)

    v_cnt   = Counter(a["violation"] for a in accs)
    t_cnt   = Counter(a["acc_type"]  for a in accs)
    time_cnt = Counter(a["acc_time"] for a in accs)
    week_cnt = Counter(a["week_type"] for a in accs)
    svr_cnt  = Counter(a["injury_svrity"] for a in accs)

    # 주말 비율
    weekend_ratio = round(week_cnt.get("주말", 0) / cnt * 100, 1) if cnt else 0

    # 전체 대비 특이 위반 (비율 차이)
    top_v = v_cnt.most_common(1)[0][0] if v_cnt else ""
    top_v_ratio = round(v_cnt.get(top_v, 0) / cnt * 100, 1)
    all_v_ratio = round(all_violation.get(top_v, 0) / total_all * 100, 1)

    result.append({
        "epdo_rank":         r["epdo_rank"],
        "link_id":           lid,
        "road_name":         r["road_name"],
        "road_rank":         r["road_rank"],
        "accident_cnt":      cnt,
        "epdo_total":        r["epdo_total"],
        "epdo_rate":         r["epdo_rate"],
        # 원인 분석
        "top_violation":     top_item(v_cnt, 3),
        "top_violation_1st": top_v,
        "top_v_ratio_pct":   top_v_ratio,
        "all_v_ratio_pct":   all_v_ratio,   # 전체 평균 비율
        "top_acc_type":      top_item(t_cnt, 2),
        "peak_time":         top_item(time_cnt, 3),
        "weekend_ratio_pct": weekend_ratio,
        "severity_dist":     top_item(svr_cnt, 4),
    })

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys())
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

print(f"\n[결과] 위험 도로 {len(result)}개 원인 분석 완료")
print(f"\n{'순위':>4} {'도로명':20s} {'사고':>4} {'epdo_rate':>12} {'주요위반'}")
print("-" * 80)
for r in result[:10]:
    print(f"  {r['epdo_rank']:>3} {r['road_name']:20s} {r['accident_cnt']:>4}건 "
          f"{float(r['epdo_rate']):>12.1f}  {r['top_violation_1st']}")

print(f"\n저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 05_GRID_EPDO_INFRA

# STEP 05 - 격자별 EPDO + 인프라 통합 분석

- 입력: output/13._교통사고_격자매핑.geojson (사고별 grid_gid + injury_svrity)
        output/격자별_통합데이터.csv (grid_gid별 인프라 집계)
- 출력: epdo_analysis/output/05_격자별_EPDO_인프라통합.csv

EPDO 가중치 출처:
  이상엽(2019). 교통사고 원인행위의 벌점 추정에 관한 연구.
  대한교통학회지, 37(5), 365-378.

In [ ]:
import csv
import json
import os
from collections import defaultdict

ACCIDENT_FILE = os.path.join(EPDO_DIR, "epdo_analysis", "output", "00_교통사고_격자매핑.geojson")
INFRA_FILE   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "00_격자별_통합데이터.csv")
OUTPUT_PATH  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "05_격자별_EPDO_인프라통합.csv")

EPDO_WEIGHTS = {
    "사망":    391,
    "중상":     69,
    "경상":      8,
    "부상신고":   6,
    "상해없음":   1,
    "기타불명":   8,
    "":          0,
}

SEVERITY_KEYS = ["사망", "중상", "경상", "부상신고", "상해없음", "기타불명"]

In [ ]:
print("=" * 60)
print("STEP 05 - 격자별 EPDO + 인프라 통합 분석")
print("=" * 60)

# 1. 사고 데이터 로드 → 격자별 EPDO 집계
print("\n[1] 사고 데이터 로드 중...")
with open(ACCIDENT_FILE, "r", encoding="utf-8") as f:
    accident_data = json.load(f)

grid_epdo = defaultdict(lambda: {
    "epdo_total":  0.0,
    "accident_cnt": 0,
    **{f"{k}_cnt": 0 for k in SEVERITY_KEYS},
})

for feat in accident_data["features"]:
    props = feat["properties"]
    gid   = props.get("grid_gid", "")
    if not gid:
        continue
    svrity = (props.get("injury_svrity") or "").strip()
    score  = EPDO_WEIGHTS.get(svrity, 0)

    grid_epdo[gid]["epdo_total"]   += score
    grid_epdo[gid]["accident_cnt"] += 1
    key = f"{svrity}_cnt" if svrity in SEVERITY_KEYS else ""
    if key:
        grid_epdo[gid][key] += 1

print(f"    사고 격자 수: {len(grid_epdo):,}개 / 전체 사고: {len(accident_data['features']):,}건")

# 2. 인프라 데이터 로드
print("\n[2] 인프라 데이터 로드 중...")
infra = {}
with open(INFRA_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        infra[row["grid_gid"]] = row

print(f"    인프라 격자 수: {len(infra):,}개")

# 3. 통합
result = []
matched = 0
for gid, epdo in grid_epdo.items():
    epdo_total  = round(epdo["epdo_total"], 2)
    acc_cnt     = epdo["accident_cnt"]
    epdo_avg    = round(epdo_total / acc_cnt, 4) if acc_cnt else 0

    inf = infra.get(gid, {})
    if inf:
        matched += 1

    result.append({
        "grid_gid":          gid,
        "epdo_total":        epdo_total,
        "epdo_avg":          epdo_avg,
        "accident_cnt":      acc_cnt,
        "사망_cnt":           epdo["사망_cnt"],
        "중상_cnt":           epdo["중상_cnt"],
        "경상_cnt":           epdo["경상_cnt"],
        "부상신고_cnt":        epdo["부상신고_cnt"],
        "상해없음_cnt":        epdo["상해없음_cnt"],
        "기타불명_cnt":        epdo["기타불명_cnt"],
        "crosswalk_cnt":     inf.get("crosswalk_cnt", 0),
        "child_zone_cnt":    inf.get("child_zone_cnt", 0),
        "school_cnt":        inf.get("school_cnt", 0),
        "kindergarten_cnt":  inf.get("kindergarten_cnt", 0),
        "daycare_cnt":       inf.get("daycare_cnt", 0),
        "bus_stop_cnt":      inf.get("bus_stop_cnt", 0),
        "cctv_cnt":          inf.get("cctv_cnt", 0),
        "cctv_cam_cnt":      inf.get("cctv_cam_cnt", 0),
        "speedbump_cnt":     inf.get("speedbump_cnt", 0),
    })

result.sort(key=lambda x: -x["epdo_total"])

# 4. 저장
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys())
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

total_epdo = sum(r["epdo_total"] for r in result)
print(f"\n[결과]")
print(f"    격자 수: {len(result):,}개 | 인프라 매칭: {matched:,}개")
print(f"    전체 EPDO 합계: {total_epdo:,.0f}점")
print(f"\n    epdo_total 상위 10개:")
print(f"  {'격자ID':12s} {'EPDO':>8} {'사고':>4} {'사망':>3} {'중상':>4} {'횡단보도':>6} {'CCTV':>5} {'과속방지턱':>7}")
print("  " + "-" * 65)
for r in result[:10]:
    print(f"  {r['grid_gid']:12s} {r['epdo_total']:>8.0f} {r['accident_cnt']:>4} "
          f"{r['사망_cnt']:>3} {r['중상_cnt']:>4} {r['crosswalk_cnt']:>6} "
          f"{r['cctv_cnt']:>5} {r['speedbump_cnt']:>7}")

print(f"\n저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 06_INFRA_ANALYSIS

# STEP 06 - 인프라 공백 분석

- 입력: epdo_analysis/output/05_격자별_EPDO_인프라통합.csv
- 출력: epdo_analysis/output/06_인프라_상관분석.csv
        epdo_analysis/output/06_인프라공백_고위험격자.csv

분석 1: EPDO × 시설물 수 상관관계 (Pearson + Spearman)
  - 어떤 시설물이 EPDO와 관련 있는지 확인
분석 2: 고위험 격자(상위 25%) 중 시설물 공백 격자 추출
  - 시설물이 없거나 전체 평균의 50% 미만인 항목을 '공백'으로 판단

In [ ]:
import csv
import math
import os

INPUT_PATH   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "05_격자별_EPDO_인프라통합.csv")
CORR_OUTPUT  = os.path.join(EPDO_DIR, "epdo_analysis", "output", "06_인프라_상관분석.csv")
GAP_OUTPUT   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "06_인프라공백_고위험격자.csv")

INFRA_COLS = [
    "crosswalk_cnt", "child_zone_cnt", "school_cnt",
    "kindergarten_cnt", "daycare_cnt", "bus_stop_cnt",
    "cctv_cnt", "cctv_cam_cnt", "speedbump_cnt",
]

# 인프라 공백 산출에 사용할 안전시설 6종
# 교육시설(school, kindergarten, daycare)은 위험 유발 시설로
# "없어도 공백이 아님" → 공백 지표에서 제외
SAFETY_COLS = [
    "crosswalk_cnt",   # 횡단보도    — 보행 안전 직결
    "child_zone_cnt",  # 어린이보호구역 — 어린이 안전 직결
    "speedbump_cnt",   # 과속방지턱  — 속도 억제
    "cctv_cnt",        # CCTV 개소  — 감시·억제
    "cctv_cam_cnt",    # CCTV 대수  — 감시·억제 보완
    "bus_stop_cnt",    # 버스정류장  — 교통약자 집산지
]
MAX_GAP = len(SAFETY_COLS)   # 6

HIGH_RISK_PERCENTILE = 75   # 상위 25%를 고위험으로 정의


# ── 통계 유틸 ────────────────────────────────────────────────────────────────

def _mean(values):
    return sum(values) / len(values) if values else 0.0


def pearson(x, y):
    n = len(x)
    if n < 2:
        return None
    mx, my = _mean(x), _mean(y)
    num = sum((xi - mx) * (yi - my) for xi, yi in zip(x, y))
    dx  = math.sqrt(sum((xi - mx) ** 2 for xi in x))
    dy  = math.sqrt(sum((yi - my) ** 2 for yi in y))
    if dx == 0 or dy == 0:
        return None
    return round(num / (dx * dy), 4)


def rank_list(values):
    """평균 순위(동점 처리 포함) 반환"""
    indexed = sorted(enumerate(values), key=lambda t: t[1])
    ranks   = [0.0] * len(values)
    i = 0
    while i < len(indexed):
        j = i
        while j < len(indexed) - 1 and indexed[j + 1][1] == indexed[j][1]:
            j += 1
        avg_rank = (i + j) / 2 + 1
        for k in range(i, j + 1):
            ranks[indexed[k][0]] = avg_rank
        i = j + 1
    return ranks


def spearman(x, y):
    return pearson(rank_list(x), rank_list(y))


def percentile(values, p):
    sv  = sorted(values)
    idx = (p / 100) * (len(sv) - 1)
    lo  = int(idx)
    hi  = min(lo + 1, len(sv) - 1)
    return sv[lo] + (idx - lo) * (sv[hi] - sv[lo])


# ── 메인 ─────────────────────────────────────────────────────────────────────

In [ ]:
print("=" * 60)
print("STEP 06 - 인프라 공백 분석")
print("=" * 60)

# 1. 데이터 로드
rows = []
with open(INPUT_PATH, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        rows.append(row)
print(f"\n[1] 격자 수: {len(rows):,}개 로드 완료")

epdo_vals = [float(r["epdo_total"]) for r in rows]

## ── 분석 1: 상관관계

In [ ]:
print("\n[2] EPDO × 시설물 상관관계 분석 중...")
corr_result = []
for col in INFRA_COLS:
    infra_vals    = [float(r.get(col, 0) or 0) for r in rows]
    pr            = pearson(epdo_vals, infra_vals)
    sr            = spearman(epdo_vals, infra_vals)
    infra_nonzero = sum(1 for v in infra_vals if v > 0)
    infra_mean    = _mean(infra_vals)

    if   (pr or 0) >  0.1: interp = "EPDO 높을수록 시설 많음 (노출 효과)"
    elif (pr or 0) < -0.1: interp = "EPDO 높을수록 시설 적음 (억제 효과 가능)"
    else:                   interp = "상관 미미"

    corr_result.append({
        "infra_col":         col,
        "pearson_r":         pr,
        "spearman_r":        sr,
        "infra_nonzero_cnt": infra_nonzero,
        "infra_mean":        round(infra_mean, 4),
        "해석":              interp,
    })

corr_result.sort(key=lambda x: abs(x["pearson_r"] or 0), reverse=True)

os.makedirs(os.path.dirname(CORR_OUTPUT), exist_ok=True)
with open(CORR_OUTPUT, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(corr_result[0].keys()))
    w.writeheader()
    w.writerows(corr_result)

print(f"\n  {'시설물':20s} {'Pearson':>9} {'Spearman':>10}  해석")
print("  " + "-" * 65)
for r in corr_result:
    print(f"  {r['infra_col']:20s} {str(r['pearson_r']):>9} {str(r['spearman_r']):>10}  {r['해석']}")

## ── 분석 2: 고위험 격자 인프라 공백

In [ ]:
print(f"\n[3] 고위험 격자 인프라 공백 추출 (상위 {100 - HIGH_RISK_PERCENTILE}%)...")
threshold = percentile(epdo_vals, HIGH_RISK_PERCENTILE)
print(f"    epdo_total 기준값: {threshold:.1f}점")

infra_means = {
    col: _mean([float(r.get(col, 0) or 0) for r in rows])
    for col in INFRA_COLS
}

gap_result = []
for r in rows:
    epdo = float(r["epdo_total"])
    if epdo < threshold:
        continue

    # 공백 산출: SAFETY_COLS 6종만 (교육시설 제외)
    gaps = []
    for col in SAFETY_COLS:
        val = float(r.get(col, 0) or 0)
        if val == 0:
            gaps.append(f"{col}(없음)")
        elif val < infra_means[col] * 0.5:
            gaps.append(f"{col}(부족)")

    gap_result.append({
        "grid_gid":          r["grid_gid"],
        "epdo_total":        epdo,
        "accident_cnt":      int(r["accident_cnt"]),
        "사망_cnt":           int(r["사망_cnt"]),
        "중상_cnt":           int(r["중상_cnt"]),
        "crosswalk_cnt":     int(float(r["crosswalk_cnt"])),
        "cctv_cnt":          int(float(r["cctv_cnt"])),
        "speedbump_cnt":     int(float(r["speedbump_cnt"])),
        "child_zone_cnt":    int(float(r["child_zone_cnt"])),
        # 학교인근 라벨 판단용 (교육시설은 공백 지표에서 제외했으나 라벨 부여에 활용)
        "school_cnt":        int(float(r.get("school_cnt", 0) or 0)),
        "kindergarten_cnt":  int(float(r.get("kindergarten_cnt", 0) or 0)),
        "gap_cnt":           len(gaps),
        "gap_items":         " | ".join(gaps) if gaps else "-",
    })

gap_result.sort(key=lambda x: (-x["gap_cnt"], -x["epdo_total"]))

with open(GAP_OUTPUT, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(gap_result[0].keys()))
    w.writeheader()
    w.writerows(gap_result)

total_high  = len(gap_result)
many_gaps   = sum(1 for r in gap_result if r["gap_cnt"] >= MAX_GAP // 2)
print(f"    고위험 격자: {total_high:,}개")
print(f"    시설물 절반 이상 공백: {many_gaps:,}개 ({many_gaps/total_high*100:.1f}%)")

print(f"\n    공백 심각 상위 10개:")
print(f"  {'격자ID':12s} {'EPDO':>7} {'사고':>4} {'공백수':>5}  공백 항목")
print("  " + "-" * 75)
for r in gap_result[:10]:
    items = r["gap_items"][:50] + ("..." if len(r["gap_items"]) > 50 else "")
    print(f"  {r['grid_gid']:12s} {r['epdo_total']:>7.0f} {r['accident_cnt']:>4}건 {r['gap_cnt']:>5}개  {items}")

print(f"\n저장: {CORR_OUTPUT}")
print(f"저장: {GAP_OUTPUT}")
print("=" * 60)

---

## 07_LINK_SPEED_CONGESTION

# STEP 07 - 링크 위험도 보강: 평균속도 + 혼잡강도

- 입력: epdo_analysis/output/03_링크별_위험도.csv
        data/09._평균속도.csv          (v_link_id, timeslot, velocity_AVRG)
        data/11._혼잡빈도강도.csv       (v_link_id, FRIN_CG)
        data/12._혼잡시간강도.csv       (v_link_id, TI_CG)
        data/08.상세도로망_네트워크.geojson (v_link_id → link_id 매핑)
- 출력: epdo_analysis/output/07_링크_속도혼잡_보강.csv

속도 보정 위험도:
  speed_adjusted_rate = epdo_rate × (avg_speed / 60.0)
  - 60km/h 기준: 초과할수록 보행자 치사율 급증 (교통안전공단 기준)
  - avg_speed < 60이면 가중치 < 1 (상대적으로 낮게 보정)

혼잡 위험도:
  congestion_risk = FRIN_CG × TI_CG / 100
  - FRIN: 혼잡빈도강도(얼마나 자주 막히는가, 0~100)
  - TI  : 혼잡시간강도(막혔을 때 얼마나 심한가, 0~100)

종합 보정 위험도:
  enhanced_rate = epdo_rate × speed_weight × (1 + congestion_weight)

In [ ]:
import csv
import json
import os
from collections import defaultdict

RISK_FILE     = os.path.join(EPDO_DIR, "epdo_analysis", "output", "03_링크별_위험도.csv")
SPEED_FILE    = os.path.join(BASE_DIR, "data", "09._평균속도.csv")
FRIN_FILE     = os.path.join(BASE_DIR, "data", "11._혼잡빈도강도.csv")
TI_FILE       = os.path.join(BASE_DIR, "data", "12._혼잡시간강도.csv")
ROAD_FILE     = os.path.join(BASE_DIR, "data", "08.상세도로망_네트워크.geojson")
OUTPUT_PATH   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "07_링크_속도혼잡_보강.csv")

SPEED_BASE    = 60.0   # km/h 기준 (보행자 치사율 급증 기준)


def build_vlink_to_link(road_file):
    """v_link_id(상행/하행) → link_id 매핑 딕셔너리 생성 (STEP 02와 동일 로직)"""
    with open(road_file, "r", encoding="utf-8") as f:
        road = json.load(f)
    mapping = {}
    for feat in road["features"]:
        p   = feat["properties"]
        lid = str(p["link_id"])
        for key in ("up_v_link", "dw_v_link"):
            v = p.get(key)
            if v:
                mapping[str(v)] = lid
    return mapping

print("=" * 60)
print("STEP 07 - 링크 위험도 보강: 속도 + 혼잡강도")
print("=" * 60)

# 1. v_link_id → link_id 매핑
print("\n[1] 도로망 매핑 로드 중...")
vlink_to_link = build_vlink_to_link(ROAD_FILE)
print(f"    v_link → link 매핑 수: {len(vlink_to_link):,}")

# 2. 평균속도 집계 (timeslot별 전체 평균, 전 시간대 합산)
print("\n[2] 평균속도 로드 중...")
speed_sum   = defaultdict(float)
speed_cnt   = defaultdict(int)
peak_speed  = defaultdict(list)   # 07~09시(출근), 16~19시(하교/퇴근)

with open(SPEED_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        vid  = row["v_link_id"]
        lid  = vlink_to_link.get(vid)
        if not lid:
            continue
        try:
            slot = int(row["timeslot"])
            spd  = float(row["velocity_AVRG"])
        except (ValueError, KeyError):
            continue
        speed_sum[lid] += spd
        speed_cnt[lid] += 1
        if slot in (7, 8, 9, 16, 17, 18, 19):   # 교통약자 위험 시간대
            peak_speed[lid].append(spd)

link_avg_speed  = {
    lid: round(speed_sum[lid] / speed_cnt[lid], 2)
    for lid in speed_sum
}
link_peak_speed = {
    lid: round(sum(v) / len(v), 2)
    for lid, v in peak_speed.items() if v
}
print(f"    속도 매칭 링크: {len(link_avg_speed):,}개")

# 3. 혼잡빈도강도 (FRIN_CG)
print("\n[3] 혼잡빈도강도 로드 중...")
frin_by_link = defaultdict(list)
with open(FRIN_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        vid = row["v_link_id"]
        lid = vlink_to_link.get(vid)
        if lid:
            try:
                frin_by_link[lid].append(float(row["FRIN_CG"]))
            except ValueError:
                pass
link_frin = {lid: round(sum(v)/len(v), 4) for lid, v in frin_by_link.items()}
print(f"    혼잡빈도 매칭 링크: {len(link_frin):,}개")

# 4. 혼잡시간강도 (TI_CG)
print("\n[4] 혼잡시간강도 로드 중...")
ti_by_link = defaultdict(list)
with open(TI_FILE, "r", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        vid = row["v_link_id"]
        lid = vlink_to_link.get(vid)
        if lid:
            try:
                ti_by_link[lid].append(float(row["TI_CG"]))
            except ValueError:
                pass
link_ti = {lid: round(sum(v)/len(v), 4) for lid, v in ti_by_link.items()}
print(f"    혼잡시간 매칭 링크: {len(link_ti):,}개")

# 5. 기존 링크 위험도 로드
print("\n[5] 기존 링크 위험도 로드 중...")
risk_rows = []
with open(RISK_FILE, "r", encoding="utf-8-sig") as f:
    risk_rows = list(csv.DictReader(f))
print(f"    링크 수: {len(risk_rows):,}개")

# 6. 보강 계산
print("\n[6] 속도·혼잡 보강 위험도 산출 중...")
result = []
matched_speed = matched_frin = matched_ti = 0

for r in risk_rows:
    lid        = r["link_id"]
    epdo_rate  = r["epdo_rate"]

    avg_spd    = link_avg_speed.get(lid)
    peak_spd   = link_peak_speed.get(lid)
    frin       = link_frin.get(lid)
    ti         = link_ti.get(lid)

    if avg_spd is not None:
        matched_speed += 1
    if frin is not None:
        matched_frin += 1
    if ti is not None:
        matched_ti += 1

    # 속도 가중치: avg_speed / 60 (60km/h 미만이면 1 미만)
    speed_weight = round(avg_spd / SPEED_BASE, 4) if avg_spd is not None else None

    # 혼잡 위험도: FRIN × TI / 100 (두 값 모두 있을 때만)
    congestion_risk = None
    if frin is not None and ti is not None:
        congestion_risk = round(frin * ti / 100, 4)

    # 종합 보정 위험도 (epdo_rate × 속도 × (1 + 혼잡))
    enhanced_rate = None
    if epdo_rate and epdo_rate != "" and speed_weight is not None:
        base = float(epdo_rate)
        cong_factor = (1 + congestion_risk / 100) if congestion_risk else 1.0
        enhanced_rate = round(base * speed_weight * cong_factor, 4)

    # 분류 레이블
    if avg_spd is not None:
        if avg_spd >= 80:
            speed_label = "고속(80+)"
        elif avg_spd >= 60:
            speed_label = "준고속(60~80)"
        elif avg_spd >= 40:
            speed_label = "중속(40~60)"
        else:
            speed_label = "저속(40미만)"
    else:
        speed_label = ""

    result.append({
        "link_id":          lid,
        "road_name":        r["road_name"],
        "road_rank":        r["road_rank"],
        "accident_cnt":     r["accident_cnt"],
        "epdo_total":       r["epdo_total"],
        "epdo_rate":        epdo_rate,
        "epdo_rank":        r["epdo_rank"],
        # 속도
        "avg_speed":        avg_spd if avg_spd is not None else "",
        "peak_speed":       peak_spd if peak_spd is not None else "",
        "speed_label":      speed_label,
        "speed_weight":     speed_weight if speed_weight is not None else "",
        # 혼잡
        "frin_cg":          frin if frin is not None else "",
        "ti_cg":            ti if ti is not None else "",
        "congestion_risk":  congestion_risk if congestion_risk is not None else "",
        # 보정 위험도
        "enhanced_rate":    enhanced_rate if enhanced_rate is not None else "",
    })

# 보정 위험도 기준 재정렬 + 순위 부여
result.sort(key=lambda x: (x["enhanced_rate"] == "", -(x["enhanced_rate"] or 0)))
rank = 1
for r in result:
    if r["enhanced_rate"] != "":
        r["enhanced_rank"] = rank
        rank += 1
    else:
        r["enhanced_rank"] = ""

# 7. 저장
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
cols = list(result[0].keys()) + ["enhanced_rank"]
cols = list(dict.fromkeys(cols))   # 중복 제거
with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols)
    w.writeheader()
    w.writerows(result)

print(f"\n[결과]")
print(f"    전체 링크: {len(result):,}개")
print(f"    속도 매칭: {matched_speed:,}개 ({matched_speed/len(result)*100:.1f}%)")
print(f"    혼잡빈도 매칭: {matched_frin:,}개 ({matched_frin/len(result)*100:.1f}%)")
print(f"    혼잡시간 매칭: {matched_ti:,}개 ({matched_ti/len(result)*100:.1f}%)")

enh = [r for r in result if r["enhanced_rate"] != ""]
print(f"    종합보정 위험도 산출: {len(enh):,}개")
print(f"\n    보정 위험도 상위 10개:")
print(f"  {'순위':>4} {'도로명':15s} {'원래rate':>12} {'속도':>6} {'혼잡':>6} {'보정rate':>14}")
print("  " + "-" * 70)
for r in result[:10]:
    if r["enhanced_rate"] == "":
        continue
    print(f"  {r['enhanced_rank']:>4} {str(r['road_name']):15s} "
          f"{float(r['epdo_rate']):>12.1f} "
          f"{str(r['avg_speed']):>6} "
          f"{str(r['congestion_risk']):>6} "
          f"{float(r['enhanced_rate']):>14.1f}")

print(f"\n저장: {OUTPUT_PATH}")
print("=" * 60)

---

## 07_ML_RISK_MODEL

# STEP 07 - ML 기반 교통사고 위험도 예측 (Random Forest)

- 입력: output/격자별_통합데이터.csv          (17,577개 전체 격자 인프라)
        epdo_analysis/output/05_격자별_EPDO_인프라통합.csv (5,723개 사고 격자 EPDO)
- 출력: epdo_analysis/output/07_ML_피처중요도.csv
        epdo_analysis/output/07_ML_위험도예측.csv

모델 설명:
  - 전체 17,577개 격자를 대상으로 학습 (사고 없는 격자 → 저위험)
  - 고위험 기준: epdo_total ≥ 69점 (전체 사고 격자 상위 25%)
  - 인프라 변수 12개를 Feature로 사용
  - class_weight='balanced': 저위험(11,854개) vs 고위험(1,480개) 불균형 보정
  - 활용: 하남교산 등 신도시의 인프라 계획 데이터만으로 사전 위험도 예측 가능

In [ ]:
import csv
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, roc_auc_score)
from sklearn.model_selection import train_test_split

GRID_FILE   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "00_격자별_통합데이터.csv")
EPDO_FILE   = os.path.join(EPDO_DIR, "epdo_analysis", "output", "05_격자별_EPDO_인프라통합.csv")
OUT_FEAT    = os.path.join(EPDO_DIR, "epdo_analysis", "output", "07_ML_피처중요도.csv")
OUT_PRED    = os.path.join(EPDO_DIR, "epdo_analysis", "output", "07_ML_위험도예측.csv")

HIGH_RISK_THRESHOLD = 69.0   # epdo_total 기준 고위험 (상위 25%)

FEATURES = [
    "crosswalk_cnt", "child_zone_cnt", "school_cnt",
    "kindergarten_cnt", "kindergarten_child_cnt", "daycare_cnt",
    "bus_stop_cnt", "cctv_cnt", "cctv_cam_cnt",
    "speedbump_cnt", "speedbump_hght_below", "speedbump_hght_above",
]

In [ ]:
print("=" * 60)
print("STEP 07 - ML 기반 교통사고 위험도 예측 (Random Forest)")
print("=" * 60)

## ── 1. 데이터 로드

In [ ]:
print("\n[1] 데이터 로드 중...")
grid_df = pd.read_csv(GRID_FILE, encoding="utf-8-sig")
epdo_df = pd.read_csv(EPDO_FILE, encoding="utf-8-sig")

print(f"    전체 격자: {len(grid_df):,}개")
print(f"    사고 격자: {len(epdo_df):,}개")

## ── 2. 데이터 병합

In [ ]:
print("\n[2] 데이터 병합 중...")

# EPDO에서 필요한 컬럼만 추출
epdo_slim = epdo_df[["grid_gid", "epdo_total"]].copy()

# LEFT JOIN: 전체 격자 + EPDO (없으면 0)
df = grid_df.merge(epdo_slim, on="grid_gid", how="left")
df["epdo_total"] = df["epdo_total"].fillna(0.0)

# 고위험 라벨 생성
df["is_high_risk"] = (df["epdo_total"] >= HIGH_RISK_THRESHOLD).astype(int)

high = df["is_high_risk"].sum()
low  = len(df) - high
print(f"    고위험(1): {high:,}개 ({high/len(df)*100:.1f}%)")
print(f"    저위험(0): {low:,}개 ({low/len(df)*100:.1f}%)")

## ── 3. Feature / Target 구성

In [ ]:
print("\n[3] 피처 구성 중...")
for col in FEATURES:
    if col not in df.columns:
        df[col] = 0
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

X = df[FEATURES].values
y = df["is_high_risk"].values

## ── 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index, test_size=0.2, random_state=42, stratify=y
)
print(f"    훈련셋: {len(X_train):,}개 | 테스트셋: {len(X_test):,}개")

## ── 5. 모델 학습

In [ ]:
print("\n[4] Random Forest 학습 중...")
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",   # 불균형 보정
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print("    학습 완료")

## ── 6. 평가

In [ ]:
print("\n[5] 모델 평가...")
y_pred      = model.predict(X_test)
y_prob      = model.predict_proba(X_test)[:, 1]
auc         = roc_auc_score(y_test, y_prob)

print(f"\n  ROC-AUC:  {auc:.4f}")
print(f"\n  분류 리포트:")
print(classification_report(y_test, y_pred,
                             target_names=["저위험", "고위험"],
                             digits=3))

cm = confusion_matrix(y_test, y_pred)
print(f"  Confusion Matrix:")
print(f"            예측저위험  예측고위험")
print(f"  실제저위험  {cm[0,0]:>7}   {cm[0,1]:>7}")
print(f"  실제고위험  {cm[1,0]:>7}   {cm[1,1]:>7}")

## ── 7. 피처 중요도 저장

In [ ]:
print("\n[6] 피처 중요도 저장 중...")
importances = model.feature_importances_
feat_rows = sorted(
    [{"feature": f, "importance": round(float(imp), 6)}
     for f, imp in zip(FEATURES, importances)],
    key=lambda x: -x["importance"]
)

os.makedirs(os.path.dirname(OUT_FEAT), exist_ok=True)
with open(OUT_FEAT, "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["rank", "feature", "importance"])
    w.writeheader()
    for i, row in enumerate(feat_rows, 1):
        w.writerow({"rank": i, **row})

print(f"\n  {'순위':>4} {'피처':25s} {'중요도':>10}")
print("  " + "-" * 44)
for i, r in enumerate(feat_rows, 1):
    bar = "#" * int(r["importance"] * 200)
    print(f"  {i:>4} {r['feature']:25s} {r['importance']:>10.4f}  {bar}")

## ── 8. 전체 격자 예측 결과 저장

In [ ]:
print("\n[7] 전체 격자 예측 중...")
all_prob  = model.predict_proba(X)[:, 1]
all_pred  = model.predict(X)

df["risk_prob"]     = np.round(all_prob, 4)
df["predicted_risk"]= all_pred

out_cols = ["grid_gid", "epdo_total", "is_high_risk", "risk_prob", "predicted_risk"]
df[out_cols].to_csv(OUT_PRED, index=False, encoding="utf-8-sig")

# 요약
pred_high = int(all_pred.sum())
print(f"    예측 고위험 격자: {pred_high:,}개 (실제: {high:,}개)")

print(f"\n  risk_prob 상위 10개 격자 (예측 위험도 높은 순):")
top10 = df.nlargest(10, "risk_prob")[
    ["grid_gid", "epdo_total", "is_high_risk", "risk_prob"]
]
print(f"  {'격자ID':12s} {'실제EPDO':>9} {'실제라벨':>6} {'위험확률':>8}")
print("  " + "-" * 42)
for _, r in top10.iterrows():
    label = "고위험" if r["is_high_risk"] else "저위험"
    print(f"  {r['grid_gid']:12s} {r['epdo_total']:>9.1f}  {label:>6}  {r['risk_prob']:>8.4f}")

print(f"\n저장: {OUT_FEAT}")
print(f"저장: {OUT_PRED}")
print("=" * 60)

---

## 08_GRID_COMPOSITE_RISK